## Random Forest Classifier

### 1. Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

### 2. Load .csv files

In [2]:
# Path to processed data
root_path = '../data/processed_data/'

# csv file names
features_kfold_csv = 'x_train_kfold.csv'
targets_kfold_csv = 'y_train_kfold.csv'
features_train_holdout_csv = 'x_train_holdout.csv'
targets_train_holdout_csv = 'y_train_holdout.csv'
features_val_holdout_csv = 'x_val_holdout.csv'
targets_val_holdout_csv = 'y_val_holdout.csv'

# Dataframes for K-fold cross validation
features_kfold = pd.read_csv(root_path + features_kfold_csv)
targets_kfold = pd.read_csv(root_path + targets_kfold_csv)

# Dataframes for holdout validation
features_train_holdout = pd.read_csv(root_path + features_train_holdout_csv)
targets_train_holdout = pd.read_csv(root_path + targets_train_holdout_csv)
features_val_holdout = pd.read_csv(root_path + features_val_holdout_csv)
targets_val_holdout = pd.read_csv(root_path + targets_val_holdout_csv)

print(features_kfold.shape)
print(targets_kfold.shape)
print(features_train_holdout.shape)
print(targets_train_holdout.shape)
print(features_val_holdout.shape)
print(targets_val_holdout.shape)

(44439, 73)
(44439, 1)
(39995, 73)
(39995, 1)
(4444, 73)
(4444, 1)


### 3. Data processing

In [3]:
# Squeeze target data
targets_kfold = targets_kfold.squeeze()
targets_train_holdout = targets_train_holdout.squeeze()
targets_val_holdout = targets_val_holdout.squeeze()

### 4. Evaluation Metrics
Helper functions for evaluating models. Metrics include: Accuracy, Precision, Recall, F1, Confusion Matrices

In [4]:
def evaluate(model, features_data, targets_data):
    prediction = model.predict(features_data)
    return {
        "Accuracy": accuracy_score(targets_data, prediction),
        "Precision": precision_score(targets_data, prediction, average="weighted", zero_division=0),
        "Recall": recall_score(targets_data, prediction, average="weighted", zero_division=0),
        "F1": f1_score(targets_data, prediction, average="weighted", zero_division=0),
        "F1_macro": f1_score(targets_data, prediction, average="macro", zero_division=0),
    }

def evaluate_class_report(model, features_data, targets_data):
    prediction = model.predict(features_data)

    return classification_report(targets_data, prediction, output_dict=True, zero_division=0)



def plot_confusion_matrix(model, features_data, targets_data, title="Confusion Matrix"):
    prediction = model.predict(features_data)
    cm = confusion_matrix(targets_data, prediction)

    fig, ax = plt.subplots(figsize=(10,10))
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)

    display.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical', values_format="d")
    plt.title(title)
    plt.tight_layout()
    plt.show()

    return cm

def plot_confusion_matrix_norm(model, features_data, targets_data, title="Confusion Matrix"):
    prediction = model.predict(features_data)
    cm_normalized = confusion_matrix(targets_data, prediction, normalize='true')

    fig, ax = plt.subplots(figsize=(10,10))
    display = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=model.classes_)

    display.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical', values_format=".2f")
    plt.title(title)
    plt.tight_layout()
    plt.show()

    return cm_normalized

# K-fold Cross Validation Evaluation
def evaluate_folds(results, scoring):
    k = len(results["test_accuracy"])

    folds = pd.DataFrame({
        name: results[f"{split}_{metric}"]
        for metric in scoring
        for split, name in [
            ("train", f"Train {metric}"),
            ("test", f"Test {metric}")
        ]
    })
    folds.index = range(1, k + 1)
    folds.index.name = "Fold"

    return folds.round(4).T

def summarize_folds(results, scoring):
    summary = pd.DataFrame({
        metric: {
            "Train Mean": results[f"train_{metric}"].mean(),
            "Train Std": results[f"train_{metric}"].std(ddof=0),
            "Test Mean": results[f"test_{metric}"].mean(),
            "Test Std": results[f"test_{metric}"].std(ddof=0),
            "Mean Gap": (results[f"train_{metric}"].mean() - results[f"test_{metric}"].mean())
        }
        for metric in scoring
    }).T
    return summary.round(4)

### 5. Model Pipelines

In [5]:
rf_model = Pipeline(steps=[
        ('clf', RandomForestClassifier())
    ])

### 6. K-fold Cross Validation

In [6]:
k = 5
kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "precision_weighted": "precision_weighted",
    "recall_macro": "recall_macro",
    "recall_weighted": "recall_weighted",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted"
}

def run_cv(model):
    return cross_validate(
        model,
        features_kfold,
        targets_kfold,
        cv=kfold,
        scoring=scoring,
        return_train_score=True
    )

In [7]:
rf_results = run_cv(rf_model)
rf_folds = evaluate_folds(rf_results, scoring)
rf_summary = summarize_folds(rf_results, scoring)

display(rf_folds)
display(rf_summary)

Fold,1,2,3,4,5
Train accuracy,1.0000,1.0000,1.0000,1.0000,1.0000
Test accuracy,0.8753,0.8690,0.8841,0.8768,0.8677
Train precision_macro,1.0000,1.0000,1.0000,1.0000,1.0000
Test precision_macro,0.8468,0.8333,0.8548,0.8476,0.8335
Train precision_weighted,1.0000,1.0000,1.0000,1.0000,1.0000
Test precision_weighted,0.8746,0.8692,0.8843,0.8753,0.8665
Train recall_macro,1.0000,1.0000,1.0000,1.0000,1.0000
Test recall_macro,0.8365,0.8220,0.8466,0.8389,0.8224
Train recall_weighted,1.0000,1.0000,1.0000,1.0000,1.0000
Test recall_weighted,0.8753,0.8690,0.8841,0.8768,0.8677


,Train Mean,Train Std,Test Mean,Test Std,Mean Gap
accuracy,1.0,0.0,0.8746,0.0059,0.1254
precision_macro,1.0,0.0,0.8432,0.0085,0.1568
precision_weighted,1.0,0.0,0.8740,0.0061,0.1260
recall_macro,1.0,0.0,0.8333,0.0096,0.1667
recall_weighted,1.0,0.0,0.8746,0.0059,0.1254
f1_macro,1.0,0.0,0.8365,0.0092,0.1635
f1_weighted,1.0,0.0,0.8724,0.0061,0.1276


##### Fine Tuning

In [10]:
Min_samples_leaf_values = [1, 2, 3, 4, 5, 6, 7, 8, 10, 20]
summary_results = {}

for value in Min_samples_leaf_values:
    model = Pipeline(steps=[
        ('clf', RandomForestClassifier(
            min_samples_leaf=value
        ))
    ])
    results = run_cv(model)
    summary_results[value] = summarize_folds(results, scoring)

df_compare = pd.concat(summary_results, names=['Min Samples Leaf', 'Metric'])
display(df_compare)

Train Mean  Train Std  Test Mean  \
Min Samples Leaf Metric                                                 
1                accuracy                1.0000     0.0000     0.8761   
                 precision_macro         1.0000     0.0000     0.8452   
                 precision_weighted      1.0000     0.0000     0.8756   
                 recall_macro            1.0000     0.0000     0.8357   
                 recall_weighted         1.0000     0.0000     0.8761   
...                                         ...        ...        ...   
20               precision_weighted      0.8619     0.0009     0.8362   
                 recall_macro            0.8027     0.0016     0.7676   
                 recall_weighted         0.8545     0.0010     0.8326   
                 f1_macro                0.8123     0.0016     0.7749   
                 f1_weighted             0.8486     0.0012     0.8243   

                                     Test Std  Mean Gap  
Min Samples Leaf Metric                                  
1                accuracy              0.0056    0.1239  
                 precision_macro       0.0079    0.1548  
                 precision_weighted    0.0059    0.1244  
                 recall_macro          0.0089    0.1643  
                 recall_weighted       0.0056    0.1239  
...                                       ...       ...  
20               precision_weighted    0.0053    0.0257  
                 recall_macro          0.0072    0.0351  
                 recall_weighted       0.0051    0.0219  
                 f1_macro              0.0070    0.0375  
                 f1_weighted           0.0054    0.0243  

[70 rows x 5 columns]

In [12]:
df_compare

Train Mean  Train Std  Test Mean  \
Min Samples Leaf Metric                                                 
1                accuracy                1.0000     0.0000     0.8761   
                 precision_macro         1.0000     0.0000     0.8452   
                 precision_weighted      1.0000     0.0000     0.8756   
                 recall_macro            1.0000     0.0000     0.8357   
                 recall_weighted         1.0000     0.0000     0.8761   
...                                         ...        ...        ...   
20               precision_weighted      0.8619     0.0009     0.8362   
                 recall_macro            0.8027     0.0016     0.7676   
                 recall_weighted         0.8545     0.0010     0.8326   
                 f1_macro                0.8123     0.0016     0.7749   
                 f1_weighted             0.8486     0.0012     0.8243   

                                     Test Std  Mean Gap  
Min Samples Leaf Metric                                  
1                accuracy              0.0056    0.1239  
                 precision_macro       0.0079    0.1548  
                 precision_weighted    0.0059    0.1244  
                 recall_macro          0.0089    0.1643  
                 recall_weighted       0.0056    0.1239  
...                                       ...       ...  
20               precision_weighted    0.0053    0.0257  
                 recall_macro          0.0072    0.0351  
                 recall_weighted       0.0051    0.0219  
                 f1_macro              0.0070    0.0375  
                 f1_weighted           0.0054    0.0243  

[70 rows x 5 columns]

In [13]:
n_estimators_values = [100, 200, 300, 400, 500]
summary_results = {}

for value in n_estimators_values:
    model = Pipeline(steps=[
        ('clf', RandomForestClassifier(
            n_estimators=value,
            min_samples_leaf=6
        ))
    ])
    results = run_cv(model)
    summary_results[value] = summarize_folds(results, scoring)

df_compare = pd.concat(summary_results, names=['Min Samples Leaf', 'Metric'])
display(df_compare)

Train Mean  Train Std  Test Mean  \
Min Samples Leaf Metric                                                 
100              accuracy                0.9147     0.0008     0.8560   
                 precision_macro         0.9143     0.0008     0.8281   
                 precision_weighted      0.9202     0.0006     0.8592   
                 recall_macro            0.8960     0.0011     0.8051   
                 recall_weighted         0.9147     0.0008     0.8560   
                 f1_macro                0.9019     0.0010     0.8113   
                 f1_weighted             0.9134     0.0009     0.8517   
200              accuracy                0.9157     0.0005     0.8564   
                 precision_macro         0.9157     0.0004     0.8285   
                 precision_weighted      0.9212     0.0005     0.8597   
                 recall_macro            0.8975     0.0006     0.8053   
                 recall_weighted         0.9157     0.0005     0.8564   
                 f1_macro                0.9033     0.0006     0.8114   
                 f1_weighted             0.9144     0.0006     0.8521   
300              accuracy                0.9162     0.0012     0.8572   
                 precision_macro         0.9165     0.0009     0.8296   
                 precision_weighted      0.9217     0.0008     0.8604   
                 recall_macro            0.8986     0.0013     0.8065   
                 recall_weighted         0.9162     0.0012     0.8572   
                 f1_macro                0.9043     0.0012     0.8127   
                 f1_weighted             0.9150     0.0012     0.8529   
400              accuracy                0.9154     0.0013     0.8573   
                 precision_macro         0.9156     0.0011     0.8296   
                 precision_weighted      0.9211     0.0009     0.8608   
                 recall_macro            0.8977     0.0013     0.8065   
                 recall_weighted         0.9154     0.0013     0.8573   
                 f1_macro                0.9034     0.0014     0.8127   
                 f1_weighted             0.9142     0.0013     0.8530   
500              accuracy                0.9158     0.0003     0.8574   
                 precision_macro         0.9163     0.0003     0.8303   
                 precision_weighted      0.9214     0.0002     0.8608   
                 recall_macro            0.8981     0.0007     0.8066   
                 recall_weighted         0.9158     0.0003     0.8574   
                 f1_macro                0.9039     0.0005     0.8130   
                 f1_weighted             0.9146     0.0003     0.8531   

                                     Test Std  Mean Gap  
Min Samples Leaf Metric                                  
100              accuracy              0.0053    0.0587  
                 precision_macro       0.0072    0.0862  
                 precision_weighted    0.0050    0.0610  
                 recall_macro          0.0083    0.0909  
                 recall_weighted       0.0053    0.0587  
                 f1_macro              0.0079    0.0906  
                 f1_weighted           0.0056    0.0617  
200              accuracy              0.0047    0.0593  
                 precision_macro       0.0064    0.0872  
                 precision_weighted    0.0044    0.0615  
                 recall_macro          0.0076    0.0922  
                 recall_weighted       0.0047    0.0593  
                 f1_macro              0.0071    0.0919  
                 f1_weighted           0.0048    0.0624  
300              accuracy              0.0054    0.0590  
                 precision_macro       0.0059    0.0869  
                 precision_weighted    0.0049    0.0613  
                 recall_macro          0.0078    0.0921  
                 recall_weighted       0.0054    0.0590  
                 f1_macro              0.0071    0.0916  
                 f1_weighted           0.0056    0.0621  
4

In [14]:
min_leaf_split = [2, 5, 10, 20, 40, 80]
summary_results = {}

for value in n_estimators_values:
    model = Pipeline(steps=[
        ('clf', RandomForestClassifier(
            n_estimators=100,
            min_samples_split=value,
            min_samples_leaf=6
        ))
    ])
    results = run_cv(model)
    summary_results[value] = summarize_folds(results, scoring)

df_compare = pd.concat(summary_results, names=['Min Samples Leaf', 'Metric'])
display(df_compare)

Train Mean  Train Std  Test Mean  \
Min Samples Leaf Metric                                                 
100              accuracy                0.8448     0.0017     0.8291   
                 precision_macro         0.8306     0.0025     0.8016   
                 precision_weighted      0.8533     0.0019     0.8341   
                 recall_macro            0.7847     0.0023     0.7605   
                 recall_weighted         0.8448     0.0017     0.8291   
                 f1_macro                0.7935     0.0023     0.7667   
                 f1_weighted             0.8373     0.0019     0.8195   
200              accuracy                0.8222     0.0024     0.8135   
                 precision_macro         0.8070     0.0027     0.7892   
                 precision_weighted      0.8327     0.0022     0.8209   
                 recall_macro            0.7487     0.0039     0.7354   
                 recall_weighted         0.8222     0.0024     0.8135   
                 f1_macro                0.7542     0.0044     0.7389   
                 f1_weighted             0.8093     0.0032     0.7991   
300              accuracy                0.8106     0.0014     0.8051   
                 precision_macro         0.7937     0.0029     0.7813   
                 precision_weighted      0.8226     0.0020     0.8144   
                 recall_macro            0.7306     0.0016     0.7221   
                 recall_weighted         0.8106     0.0014     0.8051   
                 f1_macro                0.7322     0.0024     0.7220   
                 f1_weighted             0.7941     0.0017     0.7874   
400              accuracy                0.8007     0.0020     0.7958   
                 precision_macro         0.7806     0.0048     0.7686   
                 precision_weighted      0.8107     0.0029     0.8028   
                 recall_macro            0.7137     0.0033     0.7057   
                 recall_weighted         0.8007     0.0020     0.7958   
                 f1_macro                0.7116     0.0049     0.7022   
                 f1_weighted             0.7799     0.0028     0.7737   
500              accuracy                0.7935     0.0011     0.7895   
                 precision_macro         0.7736     0.0047     0.7632   
                 precision_weighted      0.8048     0.0027     0.7983   
                 recall_macro            0.7018     0.0015     0.6954   
                 recall_weighted         0.7935     0.0011     0.7895   
                 f1_macro                0.6980     0.0025     0.6905   
                 f1_weighted             0.7709     0.0016     0.7661   

                                     Test Std  Mean Gap  
Min Samples Leaf Metric                                  
100              accuracy              0.0048    0.0157  
                 precision_macro       0.0054    0.0290  
                 precision_weighted    0.0048    0.0192  
                 recall_macro          0.0068    0.0242  
                 recall_weighted       0.0048    0.0157  
                 f1_macro              0.0061    0.0268  
                 f1_weighted           0.0050    0.0178  
200              accuracy              0.0042    0.0086  
                 precision_macro       0.0056    0.0178  
                 precision_weighted    0.0046    0.0118  
                 recall_macro          0.0055    0.0133  
                 recall_weighted       0.0042    0.0086  
                 f1_macro              0.0056    0.0153  
                 f1_weighted           0.0049    0.0101  
300              accuracy              0.0025    0.0055  
                 precision_macro       0.0061    0.0124  
                 precision_weighted    0.0039    0.0082  
                 recall_macro          0.0034    0.0085  
                 recall_weighted       0.0025    0.0055  
                 f1_macro              0.0030    0.0102  
                 f1_weighted           0.0030    0.0067  
4

In [15]:
min_leaf_splits = [2, 5, 10, 20, 40, 80]
summary_results = {}

for value in min_leaf_splits:
    model = Pipeline(steps=[
        ('clf', RandomForestClassifier(
            n_estimators=100,
            min_samples_split=value,
            min_samples_leaf=6
        ))
    ])
    results = run_cv(model)
    summary_results[value] = summarize_folds(results, scoring)

df_compare = pd.concat(summary_results, names=['Min Samples Leaf', 'Metric'])
display(df_compare)

Train Mean  Train Std  Test Mean  \
Min Samples Leaf Metric                                                 
2                accuracy                0.9156     0.0010     0.8574   
                 precision_macro         0.9149     0.0007     0.8296   
                 precision_weighted      0.9209     0.0006     0.8606   
                 recall_macro            0.8970     0.0009     0.8066   
                 recall_weighted         0.9156     0.0010     0.8574   
                 f1_macro                0.9027     0.0009     0.8128   
                 f1_weighted             0.9143     0.0011     0.8532   
5                accuracy                0.9152     0.0008     0.8568   
                 precision_macro         0.9147     0.0010     0.8287   
                 precision_weighted      0.9206     0.0007     0.8598   
                 recall_macro            0.8967     0.0004     0.8059   
                 recall_weighted         0.9152     0.0008     0.8568   
                 f1_macro                0.9024     0.0007     0.8121   
                 f1_weighted             0.9139     0.0009     0.8525   
10               accuracy                0.9152     0.0013     0.8565   
                 precision_macro         0.9151     0.0011     0.8277   
                 precision_weighted      0.9207     0.0009     0.8597   
                 recall_macro            0.8969     0.0013     0.8052   
                 recall_weighted         0.9152     0.0013     0.8565   
                 f1_macro                0.9027     0.0013     0.8112   
                 f1_weighted             0.9140     0.0014     0.8523   
20               accuracy                0.9003     0.0013     0.8524   
                 precision_macro         0.8963     0.0010     0.8238   
                 precision_weighted      0.9063     0.0010     0.8560   
                 recall_macro            0.8731     0.0014     0.7988   
                 recall_weighted         0.9003     0.0013     0.8524   
                 f1_macro                0.8804     0.0013     0.8053   
                 f1_weighted             0.8985     0.0014     0.8477   
40               accuracy                0.8767     0.0017     0.8448   
                 precision_macro         0.8657     0.0018     0.8152   
                 precision_weighted      0.8829     0.0016     0.8481   
                 recall_macro            0.8349     0.0021     0.7858   
                 recall_weighted         0.8767     0.0017     0.8448   
                 f1_macro                0.8439     0.0019     0.7927   
                 f1_weighted             0.8734     0.0018     0.8388   
80               accuracy                0.8521     0.0009     0.8329   
                 precision_macro         0.8381     0.0011     0.8052   
                 precision_weighted      0.8599     0.0009     0.8373   
                 recall_macro            0.7959     0.0019     0.7666   
                 recall_weighted         0.8521     0.0009     0.8329   
                 f1_macro                0.8052     0.0018     0.7738   
                 f1_weighted             0.8459     0.0011     0.8245   

                                     Test Std  Mean Gap  
Min Samples Leaf Metric                                  
2                accuracy              0.0049    0.0581  
                 precision_macro       0.0059    0.0852  
                 precision_weighted    0.0045    0.0603  
                 recall_macro          0.0068    0.0904  
                 recall_weighted       0.0049    0.0581  
                 f1_macro              0.0066    0.0899  
                 f1_weighted           0.0050    0.0611  
5                accuracy              0.0051    0.0583  
                 precision_macro       0.0075    0.0860  
                 precision_weighted    0.0052    0.0608  
                 recall_macro          0.0079    0.0908  
                 recall_weighted       0.0051    0.0583  
            

In [16]:
max_depths = [None, 32, 16, 8]
summary_results = {}

for value in max_depths:
    model = Pipeline(steps=[
        ('clf', RandomForestClassifier(
            n_estimators=100,
            max_depth=value,
            min_samples_split=20,
            min_samples_leaf=6
        ))
    ])
    results = run_cv(model)
    summary_results[value] = summarize_folds(results, scoring)

df_compare = pd.concat(summary_results, names=['Max Depth', 'Metric'])
display(df_compare)

Train Mean  Train Std  Test Mean  Test Std  \
Max Depth Metric                                                           
NaN       accuracy                0.9010     0.0009     0.8536    0.0050   
          precision_macro         0.8961     0.0005     0.8248    0.0068   
          precision_weighted      0.9066     0.0006     0.8565    0.0048   
          recall_macro            0.8735     0.0009     0.7998    0.0073   
          recall_weighted         0.9010     0.0009     0.8536    0.0050   
          f1_macro                0.8808     0.0007     0.8065    0.0072   
          f1_weighted             0.8993     0.0009     0.8488    0.0053   
32.0      accuracy                0.9001     0.0013     0.8530    0.0065   
          precision_macro         0.8950     0.0010     0.8238    0.0086   
          precision_weighted      0.9057     0.0009     0.8559    0.0063   
          recall_macro            0.8725     0.0010     0.7989    0.0096   
          recall_weighted         0.9001     0.0013     0.8530    0.0065   
          f1_macro                0.8797     0.0011     0.8055    0.0094   
          f1_weighted             0.8982     0.0014     0.8481    0.0068   
16.0      accuracy                0.8614     0.0024     0.8325    0.0048   
          precision_macro         0.8724     0.0008     0.8203    0.0074   
          precision_weighted      0.8781     0.0012     0.8428    0.0053   
          recall_macro            0.8095     0.0045     0.7647    0.0069   
          recall_weighted         0.8614     0.0024     0.8325    0.0048   
          f1_macro                0.8267     0.0035     0.7773    0.0073   
          f1_weighted             0.8569     0.0025     0.8239    0.0053   
8.0       accuracy                0.8005     0.0017     0.7934    0.0026   
          precision_macro         0.8344     0.0017     0.8107    0.0064   
          precision_weighted      0.8396     0.0014     0.8244    0.0048   
          recall_macro            0.6999     0.0024     0.6895    0.0042   
          recall_weighted         0.8005     0.0017     0.7934    0.0026   
          f1_macro                0.7150     0.0032     0.7010    0.0043   
          f1_weighted             0.7835     0.0022     0.7741    0.0028   

                              Mean Gap  
Max Depth Metric                        
NaN       accuracy              0.0475  
          precision_macro       0.0713  
          precision_weighted    0.0501  
          recall_macro          0.0738  
          recall_weighted       0.0475  
          f1_macro              0.0743  
          f1_weighted           0.0504  
32.0      accuracy              0.0471  
          precision_macro       0.0712  
          precision_weighted    0.0498  
          recall_macro          0.0736  
          recall_weighted       0.0471  
          f1_macro              0.0742  
          f1_weighted           0.0501  
16.0      accuracy              0.0289  
          precision_macro       0.0520  
          precision_weighted    0.0353  
          recall_macro          0.0447  
          recall_weighted       0.0289  
          f1_macro              0.0495  
          f1_weighted           0.0330  
8.0       accuracy              0.0071  
          precision_macro       0.0236  
          precision_weighted    0.0152  
          recall_macro          0.0104  
          recall_weighted       0.0071  
          f1_macro              0.0139  
          f1_weighted           0.0094

In [18]:
class_weights = [None, 'balanced', 'balanced_subsample']
summary_results = {}

for value in class_weights:
    model = Pipeline(steps=[
        ('clf', RandomForestClassifier(
            n_estimators=100,
            min_samples_split=32,
            min_samples_leaf=6
        ))
    ])
    results = run_cv(model)
    summary_results[value] = summarize_folds(results, scoring)

df_compare = pd.concat(summary_results, names=['Class Weight', 'Metric'])
display(df_compare)

Train Mean  Train Std  Test Mean  \
Class Weight       Metric                                                 
NaN                accuracy                0.8846     0.0012     0.8480   
                   precision_macro         0.8752     0.0012     0.8184   
                   precision_weighted      0.8904     0.0010     0.8510   
                   recall_macro            0.8472     0.0012     0.7905   
                   recall_weighted         0.8846     0.0012     0.8480   
                   f1_macro                0.8558     0.0013     0.7976   
                   f1_weighted             0.8819     0.0012     0.8425   
balanced           accuracy                0.8832     0.0011     0.8473   
                   precision_macro         0.8743     0.0015     0.8180   
                   precision_weighted      0.8895     0.0010     0.8507   
                   recall_macro            0.8458     0.0015     0.7898   
                   recall_weighted         0.8832     0.0011     0.8473   
                   f1_macro                0.8544     0.0014     0.7967   
                   f1_weighted             0.8804     0.0011     0.8417   
balanced_subsample accuracy                0.8845     0.0009     0.8483   
                   precision_macro         0.8750     0.0014     0.8183   
                   precision_weighted      0.8904     0.0007     0.8513   
                   recall_macro            0.8471     0.0013     0.7909   
                   recall_weighted         0.8845     0.0009     0.8483   
                   f1_macro                0.8556     0.0012     0.7978   
                   f1_weighted             0.8818     0.0009     0.8429   

                                       Test Std  Mean Gap  
Class Weight       Metric                                  
NaN                accuracy              0.0050    0.0366  
                   precision_macro       0.0074    0.0568  
                   precision_weighted    0.0051    0.0395  
                   recall_macro          0.0084    0.0567  
                   recall_weighted       0.0050    0.0366  
                   f1_macro              0.0079    0.0582  
                   f1_weighted           0.0052    0.0393  
balanced           accuracy              0.0046    0.0359  
                   precision_macro       0.0057    0.0563  
                   precision_weighted    0.0046    0.0388  
                   recall_macro          0.0074    0.0560  
                   recall_weighted       0.0046    0.0359  
                   f1_macro              0.0069    0.0577  
                   f1_weighted           0.0050    0.0387  
balanced_subsample accuracy              0.0046    0.0362  
                   precision_macro       0.0063    0.0567  
                   precision_weighted    0.0047    0.0391  
                   recall_macro          0.0070    0.0562  
                   recall_weighted       0.0046    0.0362  
                   f1_macro              0.0068    0.0578  
                   f1_weighted           0.0049    0.0389